# 🚀 Hybrid RAG PDF Summarizer (LangChain + LangGraph + Groq)
Dense + Sparse Retrieval | Llama 3.1 | Gradio UI

In [ ]:
# 1. Install Libraries
!pip install -q langchain langgraph langchain-community langchain-core
!pip install -q pypdf faiss-cpu chromadb rank_bm25
!pip install -q sentence-transformers gradio groq

In [ ]:
# 2. Imports
import os
import gradio as gr
from typing import TypedDict, List

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.prompts import PromptTemplate

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from rank_bm25 import BM25Okapi

from langgraph.graph import StateGraph, END

from groq import Groq

In [ ]:
# 3. Groq API Key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
# 4. Load and Split PDF
def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(docs)
    return chunks

In [ ]:
# 5. Dense Retriever (FAISS)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

def build_dense_retriever(docs):
    vectorstore = FAISS.from_documents(docs, embedding_model)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    return retriever, vectorstore

In [ ]:
# 6. Sparse Retriever (BM25)
def build_sparse_retriever(docs):
    texts = [doc.page_content for doc in docs]
    tokenized = [t.split() for t in texts]

    bm25 = BM25Okapi(tokenized)
    return bm25, texts

In [ ]:
# 7. Hybrid Retrieval
def hybrid_retrieve(query, dense_retriever, bm25, texts, k=5):
    dense_docs = dense_retriever.get_relevant_documents(query)

    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    sparse_docs = [Document(page_content=texts[i]) for i in top_idx]

    combined = dense_docs + sparse_docs
    unique_docs = list({doc.page_content: doc for doc in combined}.values())

    return unique_docs[:k]

In [ ]:
# 8. LLM Call (Groq)
def call_llm(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

In [ ]:
# 9. LangGraph State
class GraphState(TypedDict):
    query: str
    docs: List[Document]
    decision: str
    answer: str

In [ ]:
# 10. Decision Node
def decision_node(state):
    prompt = f"""
    Decide if the user wants:
    summary or detailed answer

    Query: {state['query']}

    Answer: summary OR detailed
    """
    decision = call_llm(prompt)
    return {"decision": decision.strip().lower()}

In [ ]:
# 11. Retrieval Node
def retrieval_node(state):
    docs = hybrid_retrieve(state["query"], DENSE_RETRIEVER, BM25, TEXTS)
    return {"docs": docs}

In [ ]:
# 12. Prompts
summary_prompt = PromptTemplate(
    input_variables=["context"],
    template="Summarize:\n{context}"
)

qa_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Answer:\nQuery: {query}\nContext: {context}"
)

In [ ]:
# 13. Nodes
def summarize_node(state):
    context = "\n\n".join([d.page_content for d in state["docs"]])
    prompt = summary_prompt.format(context=context)
    return {"answer": call_llm(prompt)}

def detailed_node(state):
    context = "\n\n".join([d.page_content for d in state["docs"]])
    prompt = qa_prompt.format(query=state["query"], context=context)
    return {"answer": call_llm(prompt)}

In [ ]:
# 14. Routing
def route(state):
    return "summarize" if "summary" in state["decision"] else "detailed"

In [ ]:
# 15. Build Graph
builder = StateGraph(GraphState)

builder.add_node("decision", decision_node)
builder.add_node("retrieve", retrieval_node)
builder.add_node("summarize", summarize_node)
builder.add_node("detailed", detailed_node)

builder.set_entry_point("decision")
builder.add_edge("decision", "retrieve")

builder.add_conditional_edges("retrieve", route, {
    "summarize": "summarize",
    "detailed": "detailed"
})

builder.add_edge("summarize", END)
builder.add_edge("detailed", END)

graph = builder.compile()

In [ ]:
# 16. Gradio UI
def process(file, query):
    global DENSE_RETRIEVER, BM25, TEXTS

    docs = load_pdf(file.name)

    DENSE_RETRIEVER, _ = build_dense_retriever(docs)
    BM25, TEXTS = build_sparse_retriever(docs)

    result = graph.invoke({"query": query})
    return result["answer"]

css = """
body { background-color: #f4f7fb; }
.title { text-align:center; font-size:30px; font-weight:bold; color:#1f4e79; }
.marquee { overflow:hidden; white-space:nowrap; }
.marquee span { display:inline-block; padding-left:100%; animation:scroll 10s linear infinite; color:#0b5ed7; }
@keyframes scroll { 0% { transform:translateX(0);} 100% { transform:translateX(-100%);} }
"""

with gr.Blocks(css=css) as demo:
    gr.HTML("""
    <div class='title'>📄 Hybrid RAG PDF System</div>
    <div class='marquee'><span>🚀 PDF Summarize by Collins Aerospace 🚀</span></div>
    """)

    file = gr.File(label="Upload PDF")
    query = gr.Textbox(label="Query")
    output = gr.Textbox(label="Response", lines=15)

    btn = gr.Button("Run")
    btn.click(process, inputs=[file, query], outputs=output)

demo.launch()